# ReasonCritic-7B — Free Colab Training with Unsloth

Train ReasonCritic-7B (verification & critique model) on **Google Colab T4 (free tier)** using **Unsloth** for 2-5x faster training with 70% less VRAM.

## 3-Stage Pipeline

| Stage | Purpose | Method | Time (T4) |
|-------|---------|--------|------------|
| 1 | Contrastive Learning | SFT on pass/fail pairs | ~3h |
| 2 | LoRA Fine-tuning | Structured critique generation | ~5h |
| 3 | DPO Alignment | Preference optimization | ~2h |

**Total: ~10 hours on Colab T4** (spread across sessions, checkpoints saved to Drive)

**Base model:** `Qwen/Qwen2.5-Coder-7B` — fits on T4 with Unsloth's 4-bit quantization

In [ ]:
#@title Cell 1: Install Unsloth & Dependencies { display-mode: "form" }
import subprocess
import sys

gpu_info = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True).strip()
print(f"GPU: {gpu_info}")

vram_mb = int(gpu_info.split(",")[1].strip().replace("MiB", "").strip())
if vram_mb < 14000:
    print(f"WARNING: Only {vram_mb}MB VRAM. 7B QLoRA on T4 works with Unsloth but batch_size=1 may be needed.")

print("Installing Unsloth and dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "unsloth[colab-new]", "--quiet"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
    "transformers>=4.46.0", "datasets>=3.0.0", "accelerate>=1.1.0",
    "peft>=0.13.0", "trl>=0.12.0", "bitsandbytes>=0.43.0",
    "scipy", "sentencepiece", "protobuf", "huggingface_hub"], check=True)

import torch
import unsloth

print(f"Unsloth {unsloth.__version__}")
print(f"PyTorch {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

In [ ]:
#@title Cell 2: Mount Google Drive { display-mode: "form" }
import os

from google.colab import drive

DRIVE_DIR = "/content/drive/MyDrive/reason-critic-7b"
drive.mount("/content/drive")
os.makedirs(f"{DRIVE_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/data", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/exports", exist_ok=True)
print(f"Project directory: {DRIVE_DIR}")

In [ ]:
#@title Cell 3: Download & Prepare Verification Data { display-mode: "form" }
import json
import os
import random

from datasets import load_dataset

DATA_DIR = f"{DRIVE_DIR}/data"
random.seed(42)

print("Downloading ReasonCritic training data...")
try:
    ds = load_dataset("fableforge/fable-5", "verification", revision="main")
    train_data = list(ds["train"])
except Exception:
    print("Generating synthetic verification data...")
    random.seed(42)
    code_snippets = [
        ("def add(a, b): return a + b", "PASS", 0.95, [], []),
        ("def divide(a, b): return a / b", "FAIL", 0.85,
         [{"type": "runtime_error", "description": "ZeroDivisionError when b=0"}],
         ["Add check: if b == 0: raise ValueError('Division by zero')"]),
        ("def get(lst, i): return lst[i]", "FAIL", 0.90,
         [{"type": "index_error", "description": "IndexError if i >= len(lst)"}],
         ["Add bounds check: if 0 <= i < len(lst): return lst[i]"]),
        ("def factorial(n): return 1 if n <= 1 else n * factorial(n-1)", "PASS", 0.97, [], []),
        ("conn = db.connect(); cursor = conn.cursor()", "FAIL", 0.80,
         [{"type": "resource_leak", "description": "Connection not closed in finally block"}],
         ["Use context manager: with db.connect() as conn:"]),
        ("import pickle; data = pickle.loads(user_input)", "FAIL", 0.95,
         [{"type": "security", "description": "Arbitrary code execution via pickle"}],
         ["Use json.loads() or restrict via RestrictedUnpickler"]),
        ("def sort_list(items): return sorted(items)", "PASS", 0.98, [], []),
        ("password = request.form['password']", "FAIL", 0.92,
         [{"type": "security", "description": "Missing input sanitization"}],
         ["Validate and sanitize input before use"]),
    ]
    train_data = []
    for code, verdict, confidence, issues, suggestions in code_snippets:
        train_data.append({
            "code": code,
            "verdict": verdict,
            "confidence": confidence,
            "issues": issues,
            "suggestions": suggestions,
        })

# Split into train/val
random.shuffle(train_data)
val_count = max(1, int(len(train_data) * 0.05))
train_split = train_data[val_count:]
val_split = train_data[:val_count]

def format_critique(example):
    code = example.get("code", "")
    verdict = example.get("verdict", "PASS")
    confidence = example.get("confidence", 0.95)
    issues = example.get("issues", [])
    suggestions = example.get("suggestions", [])
    issue_text = "\n".join(f"  - [{i.get('type', 'unknown')}] {i.get('description', '')}" for i in issues[:3]) if issues else "  None found"
    suggestion_text = "\n".join(f"  - {s}" for s in suggestions[:3]) if suggestions else "  None"
    prompt = f"<|critic|>\nAnalyze this code for correctness, security, and best practices:\n\n```\n{code[:2048]}\n```\n\nProvide a structured verification:\n- Verdict: {verdict}\n- Confidence: {confidence:.2f}\n- Issues:\n{issue_text}\n- Suggestions:\n{suggestion_text}\n</|critic|>"
    return {"text": prompt}

# Format and save data
critic_train = [format_critique(ex) for ex in train_split]
critic_val = [format_critique(ex) for ex in val_split]

with open(os.path.join(DATA_DIR, "critic_train.jsonl"), "w") as f:
    for item in critic_train:
        f.write(json.dumps(item) + "\n")
with open(os.path.join(DATA_DIR, "critic_val.jsonl"), "w") as f:
    for item in critic_val:
        f.write(json.dumps(item) + "\n")

# Generate DPO pairs
dpo_pairs = []
random.seed(42)
for ex in train_split[:min(len(train_split), 30000)]:
    code = ex.get("code", "")
    verdict = ex.get("verdict", "PASS")
    confidence = ex.get("confidence", 0.95)
    issues = ex.get("issues", [])
    suggestions = ex.get("suggestions", [])
    chosen = json.dumps({"verdict": verdict, "confidence": confidence, "issues": issues[:3], "suggestions": suggestions[:3]})
    if verdict == "PASS":
        wrong_verdict = "FAIL"
        wrong_conf = round(0.3 + random.random() * 0.3, 2)
        wrong_issues = [{"type": "style_issue", "description": "Minor concern (false positive)"}]
    else:
        wrong_verdict = "PASS"
        wrong_conf = round(0.7 + random.random() * 0.25, 2)
        wrong_issues = []
    rejected = json.dumps({"verdict": wrong_verdict, "confidence": wrong_conf, "issues": wrong_issues, "suggestions": []})
    dpo_pairs.append({"prompt": f"Analyze this code:\n```\n{code[:2048]}\n```", "chosen": chosen, "rejected": rejected})

with open(os.path.join(DATA_DIR, "dpo_train.jsonl"), "w") as f:
    for pair in dpo_pairs:
        f.write(json.dumps(pair) + "\n")

print(f"Training: {len(critic_train)} examples")
print(f"Validation: {len(critic_val)} examples")
print(f"DPO pairs: {len(dpo_pairs)}")

In [ ]:
#@title Cell 4: Stage 1 — Contrastive Learning (pass/fail classification) { display-mode: "form" }
#@markdown Pre-train with verification pairs to learn PASS/FAIL patterns.
#@markdown - LoRA r=32, alpha=64, lr=5e-4, 2 epochs
#@markdown - ~3 hours on T4

import gc
import os

import torch
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel

BASE_MODEL = "Qwen/Qwen2.5-Coder-7B"
STAGE1_OUTPUT = f"{DRIVE_DIR}/checkpoints/stage1_contrastive"

print("=" * 60)
print("  Stage 1: Contrastive Pretraining")
print("  LoRA r=32, alpha=64, lr=5e-4")
print("=" * 60)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=4096,
    load_in_4bit=True,
    dtype=None,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = FastLanguageModel.get_peft_model(
    model,
    r=32, lora_alpha=64, lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model.print_trainable_parameters()

train_ds = load_dataset("json", data_files=os.path.join(DATA_DIR, "critic_train.jsonl"), split="train")
val_ds = load_dataset("json", data_files=os.path.join(DATA_DIR, "critic_val.jsonl"), split="train")
print(f"Training: {len(train_ds)} | Validation: {len(val_ds)}")

training_args = SFTConfig(
    output_dir=STAGE1_OUTPUT,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.06,
    fp16=True,
    logging_steps=10,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    eval_strategy="steps",
    eval_steps=200,
    report_to="none",
    max_seq_length=4096,
    dataset_text_field="text",
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    weight_decay=0.01,
    seed=42,
)

trainer = SFTTrainer(
    model=model, args=training_args,
    train_dataset=train_ds, eval_dataset=val_ds,
    processing_class=tokenizer,
)

trainer.train()
trainer.save_model(os.path.join(STAGE1_OUTPUT, "final"))
tokenizer.save_pretrained(os.path.join(STAGE1_OUTPUT, "final"))

drive_ckpt = f"{DRIVE_DIR}/checkpoints/stage1_contrastive_final"
trainer.save_model(drive_ckpt)
tokenizer.save_pretrained(drive_ckpt)
print(f"\nStage 1 complete! Saved to {drive_ckpt}")

In [ ]:
#@title Cell 5: Stage 2 — LoRA Fine-tuning (structured critique) { display-mode: "form" }
#@markdown Fine-tune on structured critique generation with verification predictions.
#@markdown - LoRA r=16, alpha=32, lr=1e-4, 3 epochs
#@markdown - ~5 hours on T4

import os

import torch
from datasets import load_dataset
from peft import PeftModel
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel

STAGE2_OUTPUT = f"{DRIVE_DIR}/checkpoints/stage2_lora"

print("=" * 60)
print("  Stage 2: LoRA Fine-tuning")
print("  LoRA r=16, alpha=32, lr=1e-4")
print("=" * 60)

try:
    del model, trainer
except: pass
gc.collect()
torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=4096,
    load_in_4bit=True,
    dtype=None,
    trust_remote_code=True,
)

# Merge stage 1 adapter
stage1_adapter = f"{DRIVE_DIR}/checkpoints/stage1_contrastive_final"
if os.path.exists(stage1_adapter):
    model = PeftModel.from_pretrained(model, stage1_adapter)
    model = model.merge_and_unload()
    print("Stage 1 adapter merged")

model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=32, lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model.print_trainable_parameters()

train_ds = load_dataset("json", data_files=os.path.join(DATA_DIR, "critic_train.jsonl"), split="train")
val_ds = load_dataset("json", data_files=os.path.join(DATA_DIR, "critic_val.jsonl"), split="train")

training_args = SFTConfig(
    output_dir=STAGE2_OUTPUT,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.06,
    fp16=True,
    logging_steps=10,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    eval_strategy="steps",
    eval_steps=200,
    report_to="none",
    max_seq_length=4096,
    dataset_text_field="text",
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    weight_decay=0.01,
    seed=42,
)

trainer = SFTTrainer(model=model, args=training_args,
                     train_dataset=train_ds, eval_dataset=val_ds,
                     processing_class=tokenizer)
trainer.train()

trainer.save_model(os.path.join(STAGE2_OUTPUT, "final"))
tokenizer.save_pretrained(os.path.join(STAGE2_OUTPUT, "final"))

drive_ckpt = f"{DRIVE_DIR}/checkpoints/stage2_lora_final"
trainer.save_model(drive_ckpt)
tokenizer.save_pretrained(drive_ckpt)
print(f"\nStage 2 complete! Saved to {drive_ckpt}")

In [ ]:
#@title Cell 6: Stage 3 — DPO Alignment (preference optimization) { display-mode: "form" }
#@markdown Align critique quality using chosen/rejected pairs.
#@markdown - LoRA r=16, alpha=32, DPO beta=0.1, lr=5e-5
#@markdown - ~2 hours on T4

import gc
import os

import torch
from datasets import load_dataset
from peft import LoraConfig, PeftModel, TaskType
from trl import DPOConfig, DPOTrainer
from unsloth import FastLanguageModel

STAGE3_OUTPUT = f"{DRIVE_DIR}/checkpoints/stage3_dpo"

print("=" * 60)
print("  Stage 3: DPO Alignment")
print("  LoRA r=16, alpha=32, beta=0.1")
print("=" * 60)

try:
    del model, trainer
except: pass
gc.collect()
torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=4096,
    load_in_4bit=True,
    dtype=None,
    trust_remote_code=True,
)

# Merge stage 2 adapter
stage2_adapter = f"{DRIVE_DIR}/checkpoints/stage2_lora_final"
if os.path.exists(stage2_adapter):
    model = PeftModel.from_pretrained(model, stage2_adapter)
    model = model.merge_and_unload()
    print("Stage 2 adapter merged")

# Reference model (base, no adapter)
ref_model, _ = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=4096,
    load_in_4bit=True,
    dtype=None,
    trust_remote_code=True,
)

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

dpo_path = os.path.join(DATA_DIR, "dpo_train.jsonl")
dpo_ds = load_dataset("json", data_files=dpo_path, split="train")
print(f"DPO pairs: {len(dpo_ds)}")

training_args = DPOConfig(
    output_dir=STAGE3_OUTPUT,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    fp16=True,
    logging_steps=10,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    report_to="none",
    max_length=4096,
    max_prompt_length=2048,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    beta=0.1,
    loss_type="sigmoid",
    seed=42,
)

trainer = DPOTrainer(
    model=model, ref_model=ref_model, args=training_args,
    train_dataset=dpo_ds, processing_class=tokenizer,
    peft_config=lora_config,
)

trainer.train()
trainer.save_model(os.path.join(STAGE3_OUTPUT, "final"))
tokenizer.save_pretrained(os.path.join(STAGE3_OUTPUT, "final"))

drive_ckpt = f"{DRIVE_DIR}/checkpoints/stage3_dpo_final"
trainer.save_model(drive_ckpt)
tokenizer.save_pretrained(drive_ckpt)
print(f"\nStage 3 (DPO) complete! Saved to {drive_ckpt}")

In [ ]:
#@title Cell 7: Quick Evaluation { display-mode: "form" }
#@markdown Quick evaluation of ReasonCritic on verification tasks.

import json
import os

import torch
from peft import PeftModel
from unsloth import FastLanguageModel

print("Loading final model for evaluation...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
    trust_remote_code=True,
)

# Merge all adapters
for stage, name in [("stage1_contrastive", 1), ("stage2_lora", 2), ("stage3_dpo", 3)]:
    adapter = f"{DRIVE_DIR}/checkpoints/{stage}_final"
    if os.path.exists(adapter):
        model = PeftModel.from_pretrained(model, adapter)
        model = model.merge_and_unload()
        print(f"Merged: {stage}")

FastLanguageModel.for_inference(model)

test_codes = [
    "def add(a, b): return a + b",
    "def divide(a, b): return a / b",
    "import pickle; data = pickle.loads(user_input)",
    "for item in items: print(item)",
]

print("\nReasonCritic Evaluation:")
print("=" * 60)
for code in test_codes:
    prompt = f"Analyze this code for correctness:\n```\n{code}\n```"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.7, top_p=0.9)
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\nCode: {code[:50]}...")
    print(f"Critic: {response[:200]}...")
    print("-" * 40)

In [ ]:
#@title Cell 8: Export to GGUF { display-mode: "form" }
#@markdown Export to GGUF for llama.cpp/ollama inference.

import gc
import os

import torch

EXPORT_DIR = f"{DRIVE_DIR}/exports"
os.makedirs(EXPORT_DIR, exist_ok=True)

try:
    del model
except: pass
gc.collect()
torch.cuda.empty_cache()

# Load in full precision and merge all adapters
from peft import PeftModel
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=4096,
    load_in_4bit=False,
    dtype=torch.bfloat16,
    trust_remote_code=True,
)

for stage, name in [("stage1_contrastive", 1), ("stage2_lora", 2), ("stage3_dpo", 3)]:
    adapter = f"{DRIVE_DIR}/checkpoints/{stage}_final"
    if os.path.exists(adapter):
        model = PeftModel.from_pretrained(model, adapter)
        model = model.merge_and_unload()
        print(f"Merged: {stage}")

# Save 16-bit merged model
merged_path = f"{EXPORT_DIR}/reason-critic-7b-merged-16bit"
model.save_pretrained(merged_path)
tokenizer.save_pretrained(merged_path)
print(f"16-bit model saved to {merged_path}")

# GGUF export
try:
    model.save_pretrained_gguf(
        f"{EXPORT_DIR}/reason-critic-7b-Q4_K_M",
        tokenizer,
        quantization_method="q4_k_m",
    )
    print("GGUF (Q4_K_M) exported")
except Exception as e:
    print(f"Unsloth GGUF failed: {e}")
    print("Use llama.cpp for manual GGUF conversion")

print(f"\nExports saved to {EXPORT_DIR}")
print("\nUsage:")
print("  llama-cli -m reason-critic-7b-Q4_K_M.gguf -p 'verify this code: def foo(): pass'")